In [8]:
import pandas as pd
import numpy as np

In [ ]:
ov = pd.DataFrame({"Year": [], "MW": [], "MWh": []})
for yr in np.arange(2015, 2025):
    nskip = 1
    if yr >= 2020:
        nskip = 2
    df = pd.read_excel("../data/eia-860/december_generator"+str(yr)+".xlsx", skiprows = nskip)
    df = df[(df["Technology"] == "Batteries") & (df["Status"] == "(OP) Operating")]
    try:
        arr = np.array(["1.5", "2.3", " ", "4.1", ""])
        # Convert to float, replacing empty strings with 0
        arr_numeric = np.where(arr == " ", 0, arr).astype(float)
        ov.loc[len(ov)] = [yr, df["Nameplate Capacity (MW)"].sum()/1000, arr_numeric.sum()/1000]
    except:
        print(str(yr) + " does not have nameplate capacity and energy capacity")

2015 does not have nameplate capacity and energy capacity
2016 does not have nameplate capacity and energy capacity
2017 does not have nameplate capacity and energy capacity
2018 does not have nameplate capacity and energy capacity
2019 does not have nameplate capacity and energy capacity
2020 does not have nameplate capacity and energy capacity
2021 does not have nameplate capacity and energy capacity
2022 does not have nameplate capacity and energy capacity
2023 does not have nameplate capacity and energy capacity
2024 does not have nameplate capacity and energy capacity


In [ ]:
df[(df["Technology"] == "Batteries") & (df["Status"] == "(OP) Operating")]

1,Entity ID,Entity Name,Plant ID,Plant Name,Google Map,Bing Map,Plant State,County,Balancing Authority Code,Sector,...,Nameplate Energy Capacity (MWh),DC Net Capacity (MW),Planned Derate Year,Planned Derate Month,Planned Derate of Summer Capacity (MW),Planned Uprate Year,Planned Uprate Month,Planned Uprate of Summer Capacity (MW),Latitude,Longitude
307,16572,Salt River Project,141,Agua Fria,Map,Map,AZ,Maricopa,SRP,Electric Utility,...,100,,,,,,,,33.5561,-112.2153
468,54802,Dynegy -Moss Landing LLC,260,Dynegy Moss Landing Power Plant Hybrid,Map,Map,CA,Monterey,CISO,IPP Non-CHP,...,1200,,,,,,,,36.804837,-121.7822
469,54802,Dynegy -Moss Landing LLC,260,Dynegy Moss Landing Power Plant Hybrid,Map,Map,CA,Monterey,CISO,IPP Non-CHP,...,400,,,,,,,,36.804837,-121.7822
470,54802,Dynegy -Moss Landing LLC,260,Dynegy Moss Landing Power Plant Hybrid,Map,Map,CA,Monterey,CISO,IPP Non-CHP,...,1400,,,,,,,,36.804837,-121.7822
652,9216,Imperial Irrigation District,389,El Centro Hybrid,Map,Map,CA,Imperial,IID,Electric Utility,...,20,,,,,,,,32.802222,-115.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26669,66569,Whetstone Power Operations LLC,68058,Prairie Center Energy LLC,Map,Map,CO,Adams,WACM,IPP Non-CHP,...,,,,,,,,,39.930281,-104.7943
26676,66580,"Old Frontier Solar III PV, LLC",68069,Old Frontier Solar III,Map,Map,MA,Franklin,ISNE,IPP Non-CHP,...,,,,,,,,,42.494046,-72.62251
26681,5504,Vistra Corp,68147,Coffeen Solar BESS,Map,Map,IL,Montgomery,MISO,IPP Non-CHP,...,,,,,,,,,39.068994,-89.39336
26683,5504,Vistra Corp,68148,Baldwin Solar BESS,Map,Map,IL,Randolph,MISO,IPP Non-CHP,...,,,,,,,,,38.203536,-89.85394


In [2]:
df = pd.read_excel("../data/august_generator2024.xlsx")
df.columns = df.iloc[1]
df.drop([0,1], inplace = True)

In [3]:
for i in df.columns:
    print(i)

Entity ID
Entity Name
Plant ID
Plant Name
Google Map
Bing Map
Plant State
County
Balancing Authority Code
Sector
Generator ID
Unit Code
Nameplate Capacity (MW)
Net Summer Capacity (MW)
Net Winter Capacity (MW)
Technology
Energy Source Code
Prime Mover Code
Operating Month
Operating Year
Planned Retirement Month
Planned Retirement Year
Status
Nameplate Energy Capacity (MWh)
DC Net Capacity (MW)
Planned Derate Year
Planned Derate Month
Planned Derate of Summer Capacity (MW)
Planned Uprate Year
Planned Uprate Month
Planned Uprate of Summer Capacity (MW)
Latitude
Longitude


In [17]:
# Balancing authorities that are ISOs:
iso_list = ["MISO", "PJM", "CISO", "ISNE", "ERCO", "NYIS", "SWPP"]
# https://www.gridinfo.com/balancing-authorities?o=plants&d=desc

In [26]:
df[(df["Technology"] == "Batteries") & (df["Balancing Authority Code"].isin(iso_list))]["Sector"].unique()

array(['IPP Non-CHP', 'Electric Utility', 'IPP CHP', 'Commercial CHP',
       'Industrial CHP', 'Commercial Non-CHP', 'Industrial Non-CHP'],
      dtype=object)

In [77]:
results = dict({"Ownership": ["All", "Utility", "IPP Non-CHP"], 
                "Number": [len(df[(df["Technology"] == "Batteries") & (df["Balancing Authority Code"].isin(iso_list))]),
                           len(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list) & (df["Sector"] == "Electric Utility")]),
                           len(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list) & (df["Sector"] == "IPP Non-CHP")])
                          ], 
                "Power (MW)": [
                    round(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list)]["Nameplate Capacity (MW)"].sum(), 2), 
                    round(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list) & (df["Sector"] == "Electric Utility")]["Nameplate Capacity (MW)"].sum(), 2),
                    round(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list) & (df["Sector"] == "IPP Non-CHP")]["Nameplate Capacity (MW)"].sum(), 2)
                ], 
                "Energy (MWh)": [
                    round(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list)]["Nameplate Energy Capacity (MWh)"].replace(" ", 0).sum(), 2), 
                    round(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list) & (df["Sector"] == "Electric Utility")]["Nameplate Energy Capacity (MWh)"].sum(), 2),
                    round(df[(df["Technology"] == "Batteries") & df["Balancing Authority Code"].isin(iso_list) & (df["Sector"] == "IPP Non-CHP")]["Nameplate Energy Capacity (MWh)"].replace(" ", 0).sum(), 2)
                ]})

In [79]:
pd.DataFrame(results)

,Ownership,Number,Power (MW),Energy (MWh)
0,All,537,16754.5,45052.0
1,Utility,74,690.3,1954.1
2,IPP Non-CHP,432,16000.2,42912.2


Nantucket

In [6]:
df[df["County"] == "Nantucket"]

1,Entity ID,Entity Name,Plant ID,Plant Name,Google Map,Bing Map,Plant State,County,Balancing Authority Code,Sector,...,Nameplate Energy Capacity (MWh),DC Net Capacity (MW),Planned Derate Year,Planned Derate Month,Planned Derate of Summer Capacity (MW),Planned Uprate Year,Planned Uprate Month,Planned Uprate of Summer Capacity (MW),Latitude,Longitude
2560,13206,Nantucket Electric Co,1615,Nantucket Hybrid,Map,Map,MA,Nantucket,ISNE,Electric Utility,...,,,,,,,,,41.2583,-70.0517
2561,13206,Nantucket Electric Co,1615,Nantucket Hybrid,Map,Map,MA,Nantucket,ISNE,Electric Utility,...,,,,,,,,,41.2583,-70.0517
2562,13206,Nantucket Electric Co,1615,Nantucket Hybrid,Map,Map,MA,Nantucket,ISNE,Electric Utility,...,48,,,,,,,,41.2583,-70.0517
